# EV Market Automation (Looker Studio Starter)
Run top-to-bottom. Outputs are saved for Looker Studio with Year/Quarter slicers and threshold flags.


In [ ]:
# 0) Imports & paths
import pandas as pd, numpy as np, sqlite3, os
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
base = "/mnt/data/ev_market_automation"  # change if needed
db_path = os.path.join(base, 'ev_market.db')
print('Base:', base)


In [ ]:
# 1) Load data (replace with real CSVs later)
ev = pd.read_csv(os.path.join(base, 'data/ev_registrations_sample.csv'))
rnd = pd.read_csv(os.path.join(base, 'data/company_rnd_sample.csv'))
auto = pd.read_csv(os.path.join(base, 'data/automation_levels_sample.csv'))
display(ev.head()); display(rnd.head()); display(auto.head())


In [ ]:
# 2) SQLite load
con = sqlite3.connect(db_path)
ev.to_sql('ev_registrations', con, if_exists='replace', index=False)
rnd.to_sql('company_rnd', con, if_exists='replace', index=False)
auto.to_sql('automation_levels', con, if_exists='replace', index=False)
print('DB created at', db_path)


In [ ]:
# 3) SQL queries (Year + Quarter)
q1 = '''SELECT year, quarter, SUM(ev_sales)*1.0/SUM(total_sales) AS ev_share
        FROM ev_registrations GROUP BY year, quarter ORDER BY year, quarter;'''
q2 = '''SELECT year, quarter, company,
               rnd_spend_usd_b/NULLIF(revenue_usd_b, 0) AS rnd_ratio
        FROM company_rnd ORDER BY company, year, quarter;'''
q3 = '''WITH latest AS (
          SELECT company, MAX(year) AS max_year FROM company_rnd GROUP BY company
        ), latest_q AS (
          SELECT c.company, c.year, MAX(quarter) AS max_quarter
          FROM company_rnd c JOIN latest l ON l.company=c.company AND l.max_year=c.year
          GROUP BY c.company, c.year
        )
        SELECT a.company, a.avg_automation_level, c.rnd_spend_usd_b, c.revenue_usd_b
        FROM automation_levels a
        JOIN latest_q q ON q.company = a.company
        JOIN company_rnd c ON c.company=q.company AND c.year=q.year AND c.quarter=q.max_quarter
        ORDER BY a.avg_automation_level DESC;'''
ev_share = pd.read_sql_query(q1, con)
rnd_ratio = pd.read_sql_query(q2, con)
auto_vs_rnd_latest = pd.read_sql_query(q3, con)
display(ev_share.head()); display(rnd_ratio.head()); display(auto_vs_rnd_latest)


In [ ]:
# 4) Threshold flags & split metrics for Looker Studio colors
EV_SHARE_THRESHOLD = 0.07   # 7%
RND_RATIO_THRESHOLD = 0.025 # 2.5%

# EV share threshold fields
ev_share['ev_share_threshold'] = EV_SHARE_THRESHOLD
ev_share['below_ev_share_threshold'] = ev_share['ev_share'] < EV_SHARE_THRESHOLD
ev_share['YearQuarterLabel'] = ev_share['year'].astype(str) + ' ' + ev_share['quarter']
ev_share['ev_share_ok'] = ev_share.apply(lambda r: r['ev_share'] if r['ev_share'] >= EV_SHARE_THRESHOLD else None, axis=1)
ev_share['ev_share_below'] = ev_share.apply(lambda r: r['ev_share'] if r['ev_share'] < EV_SHARE_THRESHOLD else None, axis=1)

# R&D ratio threshold fields
rnd_ratio['rnd_ratio_threshold'] = RND_RATIO_THRESHOLD
rnd_ratio['below_rnd_threshold'] = rnd_ratio['rnd_ratio'] < RND_RATIO_THRESHOLD
rnd_ratio['rnd_ratio_ok'] = rnd_ratio.apply(lambda r: r['rnd_ratio'] if r['rnd_ratio'] >= RND_RATIO_THRESHOLD else None, axis=1)
rnd_ratio['rnd_ratio_below'] = rnd_ratio.apply(lambda r: r['rnd_ratio'] if r['rnd_ratio'] < RND_RATIO_THRESHOLD else None, axis=1)

display(ev_share.head()); display(rnd_ratio.head())


In [ ]:
# 5) Forecast (quarterly)
def yq_to_float(y, q):
    q_num = int(q[1])
    return y + (q_num-1)/4.0
X = np.array([yq_to_float(y,q) for y,q in zip(ev_share['year'], ev_share['quarter'])]).reshape(-1,1)
y = ev_share['ev_share'].values
model = LinearRegression().fit(X, y)
future = []
last_year = ev_share['year'].max()
for year in range(last_year+1, 2031):
    for q in ['Q1','Q2','Q3','Q4']:
        future.append({'year': year, 'quarter': q, 'x': yq_to_float(year,q)})
future_df = pd.DataFrame(future)
future_df['ev_share_pred'] = model.predict(future_df[['x']])
display(future_df.head())

plt.figure()
plt.plot(X.flatten(), y, marker='o', label='Actual')
plt.plot(future_df['x'], future_df['ev_share_pred'], marker='o', label='Predicted')
plt.title('EV Share: Actual vs Forecast (Quarterly)')
plt.xlabel('Year.quarter'); plt.ylabel('EV Share'); plt.legend(); plt.show()


In [ ]:
# 6) Export CSVs for Looker Studio
out_dir = os.path.join(base, 'lookerstudio'); os.makedirs(out_dir, exist_ok=True)
ev_share.to_csv(os.path.join(out_dir, 'ev_share_by_yq.csv'), index=False)
rnd_ratio.to_csv(os.path.join(out_dir, 'rnd_ratio_by_company_yq.csv'), index=False)
auto_vs_rnd_latest.to_csv(os.path.join(out_dir, 'automation_vs_rnd_latest.csv'), index=False)
print('Exported Looker Studio CSVs to', out_dir)


## (Optional) Publish to Google Sheets for auto-refresh
Uncomment and run the cell below to push the three tables to a Google Sheet, which you can use as a live Looker Studio source.

```
!pip install --quiet gspread gspread_dataframe oauth2client
from google.colab import auth
import gspread
from gspread_dataframe import set_with_dataframe
auth.authenticate_user()
gc = gspread.authorize(auth.credentials)
sh = gc.create('EV Market Automation – Data')
sh.share('your_gmail_here@gmail.com', perm_type='user', role='writer')
for title, df in [
    ('ev_share_by_yq', ev_share),
    ('rnd_ratio_by_company_yq', rnd_ratio),
    ('automation_vs_rnd_latest', auto_vs_rnd_latest)
]:
    try:
        ws = sh.add_worksheet(title=title, rows=str(len(df)+10), cols=str(len(df.columns)+10))
    except:
        ws = sh.worksheet(title)
    ws.clear(); set_with_dataframe(ws, df)
print('Google Sheet:', sh.url)
```
